# Fabric IQ supply-chain demo

Generated by build_notebook.py; do not edit generated data functions.
Use a dedicated, schema-enabled Lakehouse in the same workspace.
Attach it as the default Lakehouse and restart the Spark session.
No cloud resources, capacities or tenant settings are created here.
Writes are sequential, not a transaction across five tables. Stop readers and graph refreshes during replacement.
This notebook has not yet been executed in Fabric.


In [ ]:
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
CONFIRMATION = ""
SCENARIO = "initial"
WRITE_MODE = "create"


In [ ]:
from collections import Counter
from datetime import datetime, timedelta, timezone


BASE_TIME = datetime(2026, 9, 14, tzinfo=timezone.utc)
WINDOW_START = "2026-09-14T00:00:00Z"
WINDOW_END = "2026-09-21T00:00:00Z"
TABLE_KEYS = {
    "suppliers": "supplierId",
    "products": "productId",
    "inventory": "inventoryId",
    "orders": "orderId",
    "order_lines": "orderLineId",
}


def build_tables(scenario="initial"):
    if scenario not in ("initial", "replenished"):
        raise ValueError("scenario must be initial or replenished")
    suppliers = [
        {"supplierId": f"S{number:02}", "supplierName": f"Demo Supplier {number}",
         "leadTimeDays": number + 1}
        for number in range(1, 4)
    ]
    products = [
        {"productId": f"P{number:02}", "supplierId": f"S{(number - 1) % 3 + 1:02}",
         "productName": f"Demo Product {number:02}"}
        for number in range(1, 13)
    ]
    orders = [
        {"orderId": f"O{number:03}", "status": "Open",
         "dueAt": (BASE_TIME + timedelta(days=number - 1, hours=12)).strftime("%Y-%m-%dT%H:%M:%SZ")}
        for number in range(1, 9)
    ]
    deficits = {1: 4, 3: 2, 5: 3}
    lines = [
        {"orderLineId": f"OL{number:03}", "orderId": f"O{(number - 1) // 2 + 1:03}",
         "productId": f"P{(number - 1) % 12 + 1:02}", "requestedQty": 10,
         "allocatedQty": 10 - deficits.get(number, 0)}
        for number in range(1, 17)
    ]
    allocated = Counter()
    for line in lines:
        allocated[line["productId"]] += line["allocatedQty"]
    inventory = [
        {"inventoryId": f"I{number:03}", "productId": product["productId"],
         "warehouseId": "W01", "onHandQty": allocated[product["productId"]],
         "snapshotAt": WINDOW_START}
        for number, product in enumerate(products, 1)
    ]
    if scenario == "replenished":
        lines[0]["allocatedQty"] += 4
        inventory[0]["onHandQty"] += 4
        inventory[0]["snapshotAt"] = "2026-09-14T01:00:00Z"
    for line in lines:
        line["shortageQty"] = max(line["requestedQty"] - line["allocatedQty"], 0)
    tables = dict(suppliers=suppliers, products=products, inventory=inventory,
                  orders=orders, order_lines=lines)
    validate_tables(tables)
    return tables


def validate_tables(tables):
    identifiers = {}
    for table_name, key in TABLE_KEYS.items():
        rows = tables[table_name]
        values = [row[key] for row in rows]
        if len(values) != len(set(values)) or any(not value for value in values):
            raise ValueError(f"Invalid or duplicate key: {table_name}.{key}")
        if any(value is None for row in rows for value in row.values()):
            raise ValueError(f"Null value in {table_name}")
        identifiers[table_name] = set(values)
    for source, column, target in (
        ("products", "supplierId", "suppliers"),
        ("inventory", "productId", "products"),
        ("order_lines", "productId", "products"),
        ("order_lines", "orderId", "orders"),
    ):
        if any(row[column] not in identifiers[target] for row in tables[source]):
            raise ValueError(f"Orphan reference: {source}.{column}")
    stock = {}
    for row in tables["inventory"]:
        quantity = row["onHandQty"]
        if type(quantity) is not int or quantity < 0 or row["productId"] in stock:
            raise ValueError("Invalid inventory quantity or duplicate product")
        stock[row["productId"]] = quantity
    allocated = Counter()
    open_orders = {row["orderId"] for row in tables["orders"] if row["status"] == "Open"}
    for line in tables["order_lines"]:
        quantities = [line[key] for key in ("requestedQty", "allocatedQty", "shortageQty")]
        if any(type(quantity) is not int or quantity < 0 for quantity in quantities):
            raise ValueError("Invalid order line quantity")
        if line["allocatedQty"] > line["requestedQty"]:
            raise ValueError("Allocation exceeds request")
        if line["shortageQty"] != line["requestedQty"] - line["allocatedQty"]:
            raise ValueError("Incorrect shortage quantity")
        if line["orderId"] in open_orders:
            allocated[line["productId"]] += line["allocatedQty"]
    if any(quantity > stock.get(product_id, 0) for product_id, quantity in allocated.items()):
        raise ValueError("Total allocations exceed stock")

In [ ]:
from datetime import datetime
import re
from uuid import UUID


OWNER_PROPERTY = "fabric_iq_poc"
OWNER_VALUE = "supplychain_v1"
SCHEMAS = {
    "suppliers": "supplierId STRING, supplierName STRING, leadTimeDays BIGINT",
    "products": "productId STRING, supplierId STRING, productName STRING",
    "inventory": "inventoryId STRING, productId STRING, warehouseId STRING, onHandQty BIGINT, snapshotAt TIMESTAMP",
    "orders": "orderId STRING, status STRING, dueAt TIMESTAMP",
    "order_lines": "orderLineId STRING, orderId STRING, productId STRING, requestedQty BIGINT, allocatedQty BIGINT, shortageQty BIGINT",
}


def target_namespace(context, workspace_id, lakehouse_id, confirmation):
    if confirmation != "WRITE_DEMO_TABLES":
        raise ValueError("Set confirmation to WRITE_DEMO_TABLES after approving the target")
    expected_workspace = UUID(workspace_id)
    expected_lakehouse = UUID(lakehouse_id)
    for key in ("currentWorkspaceId", "defaultLakehouseWorkspaceId"):
        if UUID(context.get(key) or "") != expected_workspace:
            raise ValueError(f"Unexpected {key}")
    if UUID(context.get("defaultLakehouseId") or "") != expected_lakehouse:
        raise ValueError("Unexpected defaultLakehouseId")
    name = context.get("defaultLakehouseName") or ""
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError("Use an alphanumeric demo lakehouse name without spaces")
    return f"`{name}`.`dbo`"


def check_existing_table(mode, table_type, detail):
    if mode == "create":
        raise ValueError("Target table already exists; create mode never overwrites")
    properties = detail.get("properties") or {}
    if (table_type != "MANAGED" or detail.get("format") != "delta"
            or properties.get(OWNER_PROPERTY) != OWNER_VALUE
            or properties.get("delta.columnMapping.mode", "none") != "none"):
        raise ValueError("Refusing to replace an unowned, external or incompatible table")


def write_fabric_tables(spark, tables, context, workspace_id, lakehouse_id,
                        confirmation="", mode="create"):
    if mode not in ("create", "replace-demo"):
        raise ValueError("mode must be create or replace-demo")
    if set(tables) != set(SCHEMAS):
        raise ValueError("Only the five demo tables can be written")
    namespace = target_namespace(context, workspace_id, lakehouse_id, confirmation)
    spark.sql(f"SHOW TABLES IN {namespace}").collect()
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    frames = {}
    existing = set()
    for table_name, schema in SCHEMAS.items():
        target = f"{namespace}.`fiq_{table_name}`"
        rows = [dict(row) for row in tables[table_name]]
        for row in rows:
            for field in ("dueAt", "snapshotAt"):
                if field in row:
                    row[field] = datetime.fromisoformat(row[field].replace("Z", "+00:00"))
        frame = spark.createDataFrame(rows, schema=schema)
        frames[table_name] = frame
        if spark.catalog.tableExists(target):
            table_type = spark.catalog.getTable(target).tableType
            detail = spark.sql(f"DESCRIBE DETAIL {target}").first().asDict()
            check_existing_table(mode, table_type, detail)
            if spark.table(target).schema.simpleString() != frame.schema.simpleString():
                raise ValueError(f"Schema mismatch: {target}")
            existing.add(table_name)
    for table_name, frame in frames.items():
        target = f"{namespace}.`fiq_{table_name}`"
        if table_name not in existing:
            spark.sql(
                f"CREATE TABLE {target} ({SCHEMAS[table_name]}) USING DELTA "
                f"TBLPROPERTIES ('{OWNER_PROPERTY}' = '{OWNER_VALUE}', "
                "'delta.columnMapping.mode' = 'none')"
            )
        frame.write.mode("overwrite").insertInto(target)
        loaded = spark.table(target)
        if (loaded.count() != frame.count()
                or loaded.exceptAll(frame).limit(1).count()
                or frame.exceptAll(loaded).limit(1).count()):
            raise RuntimeError(f"Read-back mismatch: {target}")
        loaded.createOrReplaceTempView(f"fiq_{table_name}")
    return namespace

In [ ]:
tables = build_tables(SCENARIO)
print({name: len(rows) for name, rows in tables.items()})
print([row for row in tables["order_lines"] if row["shortageQty"]])


## Explicit write gate

Fill both IDs and set CONFIRMATION to `WRITE_DEMO_TABLES` only after checking the target.
`create` refuses existing tables. To replenish use SCENARIO=`replenished`, WRITE_MODE=`replace-demo`.
To reset use SCENARIO=`initial`, WRITE_MODE=`replace-demo`. Replacement requires the ownership property.
Never run multiple writers or change Lakehouse names/schema during this notebook.


In [ ]:
import notebookutils

namespace = write_fabric_tables(
    spark, tables, dict(notebookutils.runtime.context),
    WORKSPACE_ID, LAKEHOUSE_ID, CONFIRMATION, WRITE_MODE
)
print(f"Verified five demo tables in {namespace}")


In [ ]:
validation_sql = "SELECT\n    orders.orderId,\n    lines.orderLineId,\n    products.productId,\n    suppliers.supplierId,\n    lines.requestedQty - lines.allocatedQty AS shortageQty\nFROM fiq_orders AS orders\nJOIN fiq_order_lines AS lines ON lines.orderId = orders.orderId\nJOIN fiq_products AS products ON products.productId = lines.productId\nJOIN fiq_suppliers AS suppliers ON suppliers.supplierId = products.supplierId\nWHERE orders.status = 'Open'\n  AND orders.dueAt >= '2026-09-14T00:00:00Z'\n  AND orders.dueAt < '2026-09-21T00:00:00Z'\n  AND lines.requestedQty > lines.allocatedQty\nORDER BY orders.orderId, lines.orderLineId;"
expected_by_scenario = {'initial': [{'orderId': 'O001', 'orderLineId': 'OL001', 'productId': 'P01', 'supplierId': 'S01', 'shortageQty': 4}, {'orderId': 'O002', 'orderLineId': 'OL003', 'productId': 'P03', 'supplierId': 'S03', 'shortageQty': 2}, {'orderId': 'O003', 'orderLineId': 'OL005', 'productId': 'P05', 'supplierId': 'S02', 'shortageQty': 3}], 'replenished': [{'orderId': 'O002', 'orderLineId': 'OL003', 'productId': 'P03', 'supplierId': 'S03', 'shortageQty': 2}, {'orderId': 'O003', 'orderLineId': 'OL005', 'productId': 'P05', 'supplierId': 'S02', 'shortageQty': 3}]}
actual = [row.asDict() for row in spark.sql(validation_sql).collect()]
if actual != expected_by_scenario[SCENARIO]:
    raise RuntimeError(f"Acceptance mismatch: {actual}")
print({"scenario": SCENARIO, "verifiedShortages": actual})


## Next: Ontology

Follow entity-mapping.md to bind the five managed Delta tables.
Refresh the ontology graph after initial ingestion, replenishment and reset.
Run the agent questions in acceptance-cases.json three times per scenario.
Passing this notebook verifies only Delta ingestion and SQL, not the ontology or agent.
